# VLM-FO1 Inference in Jupyter Notebook

This notebook demonstrates VLM-FO1 usage with unified dependencies that work across all platforms.

**Works on:**
- Local Jupyter
- Google Colab
- Kaggle Notebooks
- SageMaker Studio
- Any Jupyter environment with GPU

## Step 1: Environment Check and Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Detect CUDA version and recommend installation
import subprocess

try:
    cuda_version = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=cuda_version', '--format=csv,noheader'],
        universal_newlines=True
    ).strip().split('\n')[0]
    print(f"✓ CUDA version detected: {cuda_version}")
    
    cuda_major = cuda_version.split('.')[0]
    if cuda_major == '11':
        print("✓ Recommended: CUDA 11.8 wheel (cu118)")
        torch_index = "https://download.pytorch.org/whl/cu118"
    elif int(cuda_major) >= 12:
        print("✓ CUDA 12.x detected - using cu118 for compatibility")
        torch_index = "https://download.pytorch.org/whl/cu118"
    else:
        print("⚠️  Old CUDA version - using CPU")
        torch_index = "https://download.pytorch.org/whl/cpu"
except:
    print("⚠️  No GPU detected - using CPU")
    torch_index = "https://download.pytorch.org/whl/cpu"

In [ ]:
# Install PyTorch with correct CUDA version
!pip install --quiet torch>=2.1,<2.7 torchvision>=0.16 --index-url {torch_index}

# Install VLM-FO1 dependencies (unified with pyproject.toml)
!pip install --quiet \
    transformers>=4.45,<5.0 \
    timm>=0.9.0 \
    accelerate>=1.0 \
    safetensors>=0.4.0 \
    pillow>=9.0 \
    numpy>=1.21

# Install VLM-FO1 from GitHub
!pip install --quiet git+https://github.com/om-ai-lab/VLM-FO1.git

print("\n✓ Installation complete!")

## Step 2: Verify Installation

In [ ]:
# Run selfcheck
!python -m vlm_fo1

In [ ]:
# Import and verify
import torch
import vlm_fo1
from vlm_fo1._backend import info
import json

print(f"VLM-FO1 version: {vlm_fo1.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Show diagnostics
diagnostics = info()
print("\nDiagnostics:")
print(json.dumps(diagnostics, indent=2))

## Step 3: Load Model

In [ ]:
from vlm_fo1.model.builder import load_pretrained_model

model_path = "omlab/VLM-FO1_Qwen2.5-VL-3B-v01"

print(f"Loading model: {model_path}")
tokenizer, model, image_processors = load_pretrained_model(
    model_path,
    device="cuda" if torch.cuda.is_available() else "cpu",
    load_8bit=False,  # Set to True for lower memory usage
)

print("✓ Model loaded successfully!")

## Step 4: Run Inference

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from vlm_fo1.mm_utils import prepare_inputs, extract_predictions_to_bboxes, draw_bboxes_and_save
from vlm_fo1.task_templates import OD_template

# Load or create test image
# Option 1: Use demo image
# !wget https://raw.githubusercontent.com/om-ai-lab/VLM-FO1/main/demo/demo_image.jpg -O test_image.jpg

# Option 2: Upload your own image (in Jupyter/Colab)
# from google.colab import files  # Uncomment for Colab
# uploaded = files.upload()  # Uncomment for Colab
# image_path = list(uploaded.keys())[0]  # Uncomment for Colab

# For this demo, use a dummy image path
image_path = "test_image.jpg"  # Replace with your image

# Display image
img = Image.open(image_path).convert("RGB")
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.title("Input Image")
plt.show()

print(f"Image size: {img.size}")

In [ ]:
# Define query and bounding boxes
query = "orange"  # Change to your target object
bbox_list = [
    [161.0, 11.0, 292.0, 127.0],
    [268.0, 61.0, 428.0, 226.0],
    [12.0, 100.0, 140.0, 227.0],
]  # Example bboxes (x1, y1, x2, y2)

# Prepare messages
messages = [{
    "role": "user",
    "content": [
        {"type": "image_url", "image_url": {"url": image_path}},
        {"type": "text", "text": OD_template.format(query)},
    ],
    "bbox_list": bbox_list,
}]

# Prepare inputs
generation_kwargs = prepare_inputs(
    model_path,
    model,
    image_processors,
    tokenizer,
    messages,
    max_tokens=512,
    top_p=0.05,
    temperature=0.0,
    do_sample=False,
)

# Run inference
print(f"Running inference for query: '{query}'")
with torch.inference_mode():
    output_ids = model.generate(**generation_kwargs)

outputs = tokenizer.decode(
    output_ids[0, generation_kwargs['inputs'].shape[1]:],
    skip_special_tokens=True
).strip()

print(f"\nModel output: {outputs}")

In [ ]:
# Extract and visualize results
predicted_bboxes = extract_predictions_to_bboxes(outputs, bbox_list)

print(f"Predicted bounding boxes: {predicted_bboxes}")

# Draw bboxes on image
result_path = "result_with_bboxes.jpg"
draw_bboxes_and_save(
    image=img,
    fo1_bboxes=predicted_bboxes,
    output_path=result_path
)

# Display result
result_img = Image.open(result_path)
plt.figure(figsize=(10, 8))
plt.imshow(result_img)
plt.axis('off')
plt.title(f"Detection Results for '{query}'")
plt.show()

print("\n✓ Inference complete!")

## Step 5: GPU Memory Profiling (Optional)

In [ ]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    max_allocated = torch.cuda.max_memory_allocated() / 1024**2
    
    print("GPU Memory Usage:")
    print(f"  Allocated: {allocated:.2f} MB")
    print(f"  Reserved: {reserved:.2f} MB")
    print(f"  Peak: {max_allocated:.2f} MB")
else:
    print("CPU mode - no GPU memory to report")

## Troubleshooting

If you encounter issues:

1. **CUDA Out of Memory:** Try `load_8bit=True` when loading the model
2. **Module not found:** Restart the kernel and re-run installation cells
3. **ABI mismatch:** Check that PyTorch CUDA version matches your system:
   ```python
   import torch
   print(torch.version.cuda)  # Should be 11.x for cu118 wheels
   ```
4. **See full diagnostics:**
   ```python
   from vlm_fo1._backend import info
   import json
   print(json.dumps(info(), indent=2))
   ```